# 05 Figures: metrics, tables and figures for the paper

Drafted with an AI assistant, see docs/ai_log.md.

Inputs: `data/processed/model_table.parquet`, `data/processed/predictions.parquet`, `data/processed/gp_cells.parquet` and `figures/gp_summary.csv`, `data/processed/segments.parquet` and `figures/segments_summary.csv` (all written by 04).
Outputs: `figures/metrics.csv`, `figures/table_model_gaps.tex`, `figures/capture_curve.pdf` (roof yield, neighbour share, LASSO and `mlp`; the heating sensitivity models and `mlp_wide` are in `metrics.csv` and the gap table) and `figures/gp_residual_map.pdf` (grid-cell map of the LASSO residuals on the test split), `figures/table_segments.tex` and `figures/pca_segments.pdf` (segments fitted in 04).

Metrics per model, split and label: capture at 5% and 10%, ROC AUC, average precision. A `random` row gives the reference values. Test rows also carry 95% bootstrap intervals (`<metric>_lo`, `<metric>_hi`).

In [ ]:
import helpers
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

In [ ]:
model_table = pd.read_parquet(helpers.PROCESSED / "model_table.parquet")
predictions = pd.read_parquet(helpers.PROCESSED / "predictions.parquet")
data = model_table.merge(predictions, on=["egid", "cutoff", "split"], how="inner")
assert len(data) == len(predictions)
# Fixed row order before scoring: ties in a score (many buildings share the same
# neighbour share) are broken by this order in the stable sorts below.
data = data.sort_values(["split", "model", "egid"]).reset_index(drop=True)
helpers.FIGURES.mkdir(exist_ok=True)

Bootstrap on the test split. Resampling unit: whole municipalities (`ggdenr`), because neighbour shares and outcomes are correlated within a municipality (docs/decisions.md). B = 200, seed 0. The replicate row indices are drawn once and used for every model, so differences between models are paired. Rows of each model are in egid order (same buildings, same order), a municipality drawn twice contributes its rows twice, and the copies of a row sit next to each other in the replicate; the stable sort in the capture metric then breaks ties as in the original data.

In [ ]:
N_BOOT = 200
SEED = 0
METRICS = ["capture_5", "capture_10", "roc_auc", "avg_precision"]


def score_metrics(y, score):
    return {
        "capture_5": helpers.capture_at_k(y, score, 0.05),
        "capture_10": helpers.capture_at_k(y, score, 0.10),
        "roc_auc": roc_auc_score(y, score),
        "avg_precision": average_precision_score(y, score),
    }


test_part = data[data["split"] == "test"]
models = sorted(test_part["model"].unique())
# Same buildings in the same order for every model, so one set of indices fits all
egids = test_part[test_part["model"] == models[0]]["egid"].to_numpy()
for model in models:
    assert np.array_equal(test_part[test_part["model"] == model]["egid"].to_numpy(), egids)
muni = test_part[test_part["model"] == models[0]]["ggdenr"].to_numpy()
replicates = helpers.cluster_bootstrap_indices(muni, N_BOOT, SEED)
sizes = [len(r) for r in replicates]
print(f"{len(set(muni))} municipalities in the test pool")
print(f"replicate size {min(sizes)} to {max(sizes)} rows")

# boot[(model, label)][metric] = array with one value per replicate
boot = {}
for model in models:
    part = test_part[test_part["model"] == model]
    score = part["score"].to_numpy()
    for label in ["y", "y_2024"]:
        y = part[label].astype(int).to_numpy()
        values = [score_metrics(y[idx], score[idx]) for idx in replicates]
        boot[(model, label)] = {m: np.array([v[m] for v in values]) for m in METRICS}

One row per (model, split, label). Labels: `y` for both splits, `y_2024` for the test split. Test rows get the 2.5th and 97.5th percentiles of the replicate values in `<metric>_lo` and `<metric>_hi`; train and `random` rows leave them empty.

In [ ]:
rows = []
for split in ["train", "test"]:
    labels = ["y"] if split == "train" else ["y", "y_2024"]
    for label in labels:
        for model, part in data[data["split"] == split].groupby("model"):
            y = part[label].astype(int).to_numpy()
            score = part["score"].to_numpy()
            rows.append(
                {
                    "model": model,
                    "split": split,
                    "label": label,
                    "n": len(y),
                    "positives": int(y.sum()),
                    "base_rate": y.mean(),
                    "capture_5": helpers.capture_at_k(y, score, 0.05),
                    "capture_10": helpers.capture_at_k(y, score, 0.10),
                    "roc_auc": roc_auc_score(y, score),
                    "avg_precision": average_precision_score(y, score),
                }
            )
        # Random reference, once per (split, label): same pool, expected values.
        # The last model row has the same n, positives and base rate as all others.
        last = rows[-1]
        rows.append(
            {
                "model": "random",
                "split": split,
                "label": label,
                "n": last["n"],
                "positives": last["positives"],
                "base_rate": last["base_rate"],
                "capture_5": 0.05,
                "capture_10": 0.10,
                "roc_auc": 0.5,
                "avg_precision": last["base_rate"],
            }
        )
metrics = pd.DataFrame(rows).sort_values(["split", "label", "model"]).reset_index(drop=True)
assert not metrics.duplicated(["model", "split", "label"]).any()
# Bootstrap intervals on the test rows (percentiles of the replicate values)
for metric in METRICS:
    metrics[f"{metric}_lo"] = np.nan
    metrics[f"{metric}_hi"] = np.nan
for (model, label), reps in boot.items():
    row = (metrics["model"] == model) & (metrics["split"] == "test") & (metrics["label"] == label)
    for metric in METRICS:
        metrics.loc[row, f"{metric}_lo"] = np.percentile(reps[metric], 2.5)
        metrics.loc[row, f"{metric}_hi"] = np.percentile(reps[metric], 97.5)
metrics = metrics.round(6)
metrics.to_csv(helpers.FIGURES / "metrics.csv", index=False, lineterminator="\n")
print(metrics.to_string())

Paired gaps on the test split: the difference between two models on the same buildings, replicate by replicate with the same municipality draws, for the headline pairs among the models present. Point estimate from the full test pool, 95% percentile interval of the replicate differences. Output: `figures/table_model_gaps.tex`.

In [ ]:
GAP_METRICS = {"capture_10": "Capture at 10%", "roc_auc": "ROC AUC"}
PAIRS = [
    ("lasso", "nbr_share"),
    ("lasso", "roof_yield"),
    ("nbr_share", "roof_yield"),
    # Heating sensitivity models against the main LASSO (leakage-contaminated upper bound)
    ("lasso_heat_source", "lasso"),
    ("lasso_heat_date", "lasso"),
    # Small neural networks against the main LASSO
    ("mlp", "lasso"),
    ("mlp_wide", "lasso"),
]
PAIRS = [(a, b) for a, b in PAIRS if a in models and b in models]
LABELS = {"y": "Test, first PV 2024-2025", "y_2024": "Test, first PV 2024 only"}


def full_value(model, label, metric):
    part = test_part[test_part["model"] == model]
    return score_metrics(part[label].astype(int).to_numpy(), part["score"].to_numpy())[metric]


# Plain LaTeX tabular with fixed formatting, to be \\input in the paper
lines = [
    "\\begin{tabular}{llrr}",
    "\\toprule",
    "Label & Gap (first minus second) & Capture at 10\\% & ROC AUC \\\\",
    "\\midrule",
]
for label, label_text in LABELS.items():
    for a, b in PAIRS:
        cells = []
        for metric in GAP_METRICS:
            point = full_value(a, label, metric) - full_value(b, label, metric)
            diff = boot[(a, label)][metric] - boot[(b, label)][metric]
            lo, hi = np.percentile(diff, [2.5, 97.5])
            cells.append(f"{point:.3f} [{lo:.3f}, {hi:.3f}]")
        name = f"{a.replace('_', ' ')} minus {b.replace('_', ' ')}"
        lines.append(f"{label_text} & {name} & " + " & ".join(cells) + " \\\\")
    if label != list(LABELS)[-1]:
        lines.append("\\midrule")
lines += ["\\bottomrule", "\\end{tabular}"]
(helpers.FIGURES / "table_model_gaps.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

Capture curve on the test split, label `y`: share of adopters found as we go down the ranking.

In [ ]:
# Main models only: the heating models use leaked information and must not look like candidates;
# `mlp_wide` is in the tables, not on the curve
CURVE_MODELS = ["roof_yield", "nbr_share", "lasso", "mlp"]
test = data[(data["split"] == "test") & data["model"].isin(CURVE_MODELS)]
fig, ax = plt.subplots(figsize=(5, 4))
for model, part in test.groupby("model"):
    y = part["y"].astype(int).to_numpy()
    order = np.argsort(-part["score"].to_numpy(), kind="stable")
    share_ranked = np.arange(1, len(y) + 1) / len(y)
    share_captured = np.cumsum(y[order]) / y.sum()
    ax.plot(share_ranked, share_captured, label=model)
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="random")
ax.set_xlabel("Share of buildings ranked (highest score first)")
ax.set_ylabel("Share of adopters captured")
ax.set_title("Capture curve, test split (first PV 2024-2025)")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(helpers.FIGURES / "capture_curve.pdf", metadata={"CreationDate": None})
plt.close(fig)

Map of the spatial structure left after the LASSO (test split). Left: observed mean residual per grid cell (`y - p_cal`, calibrated LASSO score, see 04). Right: posterior mean of the Gaussian process fitted in 04. Both panels share one diverging colour scale centred on zero (limit: 99th percentile of the absolute observed cell residuals, values beyond it are clipped). Only cells with at least 10 pool buildings are drawn; there are no per-building points.

In [ ]:
gp_cells = pd.read_parquet(helpers.PROCESSED / "gp_cells.parquet")
gp_summary = pd.read_csv(helpers.FIGURES / "gp_summary.csv").set_index("split")
cells = gp_cells[gp_cells["split"] == "test"]
side = int(gp_summary.loc["test", "cell_size_m"])
assert cells["n"].min() >= 10 and cells[["mean_residual", "gp_mean"]].notna().all().all()

# Fill a regular array (rows = northing, columns = easting); empty cells stay masked
col = ((cells["cell_x"] - cells["cell_x"].min()) // side).astype(int).to_numpy()
row = ((cells["cell_y"] - cells["cell_y"].min()) // side).astype(int).to_numpy()
extent = [
    cells["cell_x"].min() / 1000, (cells["cell_x"].max() + side) / 1000,
    cells["cell_y"].min() / 1000, (cells["cell_y"].max() + side) / 1000,
]  # fmt: skip
limit = np.percentile(np.abs(cells["mean_residual"]), 99)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.4), sharex=True, sharey=True)
for ax, column, title in [
    (axes[0], "mean_residual", "Observed cell mean residual"),
    (axes[1], "gp_mean", "GP posterior mean"),
]:
    grid = np.full((row.max() + 1, col.max() + 1), np.nan)
    grid[row, col] = cells[column].to_numpy()
    image = ax.imshow(
        np.ma.masked_invalid(grid), origin="lower", extent=extent, cmap="RdBu_r",
        vmin=-limit, vmax=limit, interpolation="nearest",
    )  # fmt: skip
    ax.set_title(title)
    ax.set_xlabel("Easting, LV95 (km)")
    ax.set_aspect("equal")
axes[0].set_ylabel("Northing, LV95 (km)")
fig.colorbar(
    image, ax=axes, shrink=0.8, extend="both", label="Observed minus calibrated probability"
)
fig.suptitle(
    f"LASSO residuals, test split (first PV 2024-2025), {side} m cells, "
    f"{len(cells)} cells with at least 10 buildings"
)
fig.savefig(helpers.FIGURES / "gp_residual_map.pdf", metadata={"CreationDate": None})
plt.close(fig)
print(gp_summary.loc["test"].to_string())

## Segments (PCA and k-means, descriptive)

Segments of the pool buildings fitted in 04 on the train split (10 standardised traits, k-means, see the section there). Outputs: `figures/table_segments.tex` and `figures/pca_segments.pdf`. The table gives size and adoption rate of each segment on both splits and the five traits that differ most between the segment centres (largest range of the standardised centre across segments); the description column is left empty on purpose. The plot shows the test split on the first two principal components as a hexagonal density (no individual buildings), with the segment centres (train means) marked.


In [ ]:
segments = pd.read_parquet(helpers.PROCESSED / "segments.parquet")
seg_summary = pd.read_csv(helpers.FIGURES / "segments_summary.csv")
trait_keys = [c.removeprefix("mean_") for c in seg_summary.columns if c.startswith("mean_")]

# The five traits with the widest spread of the standardised centre across segments
spread = pd.Series({k: np.ptp(seg_summary[f"z_{k}"]) for k in trait_keys})
shown = spread.sort_values(ascending=False, kind="stable").index[:5].tolist()
print("most distinguishing traits:", shown)

# Header text and number format per trait (means in original units, train)
TRAIT_FORMAT = {
    "area": ("Suitable roof (m$^2$)", "{:,.0f}"),
    "yield": ("Yield (MWh/yr)", "{:,.0f}"),
    "share": ("Suitable share of roof", "{:.2f}"),
    "class": ("Best roof class", "{:.1f}"),
    "footprint": ("Footprint (m$^2$)", "{:,.0f}"),
    "floors": ("Floors", "{:.1f}"),
    "dwellings": ("Dwellings", "{:.1f}"),
    "period": ("Period rank (1 to 13)", "{:.1f}"),
    "nbr_share": ("PV share, 250 m", "{:.2f}"),
    "nbr_n": ("Buildings within 250 m", "{:.0f}"),
}


def thousands(text):
    # LaTeX thousands separator, as in table_pool_filters.tex
    return text.replace(",", "{,}")


lines = [
    "\\begin{tabular}{llrrrr" + "r" * len(shown) + "}",
    "\\toprule",
    " & & \\multicolumn{2}{c}{Train (cut-off 2021)} & \\multicolumn{2}{c}{Test (cut-off 2023)}"
    + f" & \\multicolumn{{{len(shown)}}}{{c}}{{Mean on train}} \\\\",
    f"\\cmidrule(lr){{3-4}} \\cmidrule(lr){{5-6}} \\cmidrule(lr){{7-{6 + len(shown)}}}",
    "Segment & Description & Buildings & Adoption (\\%) & Buildings & Adoption (\\%) & "
    + " & ".join(TRAIT_FORMAT[k][0] for k in shown)
    + " \\\\",
    "\\midrule",
]
for row in seg_summary.itertuples():
    cells = [
        str(int(row.segment)),
        "",
        thousands(f"{int(row.train_n):,}"),
        f"{100 * row.train_rate:.1f}",
        thousands(f"{int(row.test_n):,}"),
        f"{100 * row.test_rate:.1f}",
    ]
    for key in shown:
        value = getattr(row, f"mean_{key}")
        value = value / 1000 if key == "yield" else value  # kWh to MWh
        cells.append(thousands(TRAIT_FORMAT[key][1].format(value)))
    lines.append(" & ".join(cells) + " \\\\")
lines += ["\\bottomrule", "\\end{tabular}"]
(helpers.FIGURES / "table_segments.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

Plot: first two principal components (fitted on train) of the test buildings as a hexagonal density with a logarithmic colour scale, and the k segment centres (train segment means projected on the same components). The axis labels give the share of the total variance of the standardised traits explained by each component, recomputed from the train scores in `segments.parquet`.


In [ ]:
train_scores = segments[segments["split"] == "train"]
n_train, n_traits = len(train_scores), len(trait_keys)
# Variance of a component over the total variance of the standardised traits (n_traits)
ratio = [train_scores[c].var(ddof=1) / (n_traits * n_train / (n_train - 1)) for c in ["pc1", "pc2"]]
centres = train_scores.groupby("segment")[["pc1", "pc2"]].mean()
test_scores = segments[segments["split"] == "test"]

fig, ax = plt.subplots(figsize=(6, 4.5))
hexes = ax.hexbin(
    test_scores["pc1"], test_scores["pc2"], gridsize=45, bins="log", mincnt=1, cmap="Greys"
)
fig.colorbar(hexes, ax=ax, label="Buildings per hexagon (log scale)")
ax.scatter(centres["pc1"], centres["pc2"], marker="X", s=90, color="tab:red", edgecolor="white")
for segment, (x, y) in centres.iterrows():
    ax.annotate(f"{segment}", (x, y), xytext=(6, 6), textcoords="offset points", color="tab:red")
ax.set_xlabel(f"PC1 ({100 * ratio[0]:.1f}% of variance)")
ax.set_ylabel(f"PC2 ({100 * ratio[1]:.1f}% of variance)")
ax.set_title(f"Test split on the first two components, {len(centres)} segment centres")
fig.tight_layout()
fig.savefig(helpers.FIGURES / "pca_segments.pdf", metadata={"CreationDate": None})
plt.close(fig)
print(f"explained variance: PC1 {ratio[0]:.3f}, PC2 {ratio[1]:.3f}")